In [ ]:
%load_ext autoreload
%autoreload 2
%load_ext jupyter_black

In [ ]:
from pathlib import Path

import lightning as L
import torch
from lightning.pytorch.loggers import TensorBoardLogger
from tabseq_torch.dl import cat_embedding_params, collate_padded_batch_fn
from tabseq_torch.dl.gpdf_dataset import PdDataset
from tabseq_torch.dl.types import ColumnType
from tabseq_torch.lit.lstm import LitMulticlassTabLSTM
from torch.utils import data

In [ ]:
_NAME_CHARS_COL = "name_chars"
_GROUP = "G"
_CAT_COLUMNS = [_NAME_CHARS_COL, "vc"]
_NUM_COLUMNS = ["F", "F_Rank", "T"]
# _NUM_COLUMNS = ["F_Rank"]
_N_PARTITIONS = 3
_LABELES_COL = "labels"
_T = "T"

In [ ]:
data_path = Path("/home/jovyan/data/tbts_data")
filename = "tbts-names"
src_parquet = data_path / f"{filename}.parquet"

In [ ]:
# data_prep(
#     src_parquet=src_parquet,
#     cat_columns=_CAT_COLUMNS,
#     group_cl_name=_GROUP,
#     T_cl_name=_T,
#     labels_col_name=_LABELES_COL,
#     overwrite=False,
# )

In [ ]:
src_parquet

In [ ]:
torch.cuda.is_available()

In [ ]:
pd_dataset = PdDataset(
    tbts_path=src_parquet,
    tbts_groupby_column=_GROUP,
    cat_columns=_CAT_COLUMNS,
    num_columns=_NUM_COLUMNS,
    targets_path=data_path / f"{filename}-labels.parquet",
    target_column=_LABELES_COL,
)

In [ ]:
pd_dataset.num_classes

In [ ]:
pd_dataset.catnum

In [ ]:
embedding_dim = 8
embedding_dims = dict.fromkeys(
    pd_dataset.get_columns_by_type(column_type=ColumnType.CAT), embedding_dim
)
embedding_dims["vc"] = 4
embedding_params = cat_embedding_params(
    catnum=pd_dataset.catnum, col_embedding_dims=embedding_dims
)
b_lstm = LitMulticlassTabLSTM(
    cat_enc_params=embedding_params,
    h_size=64,
    num_classes=pd_dataset.num_classes,
    columns_by_type=pd_dataset.type_columns,
    num_layers=4,
    lr=1e-4,
)

In [ ]:
train_set_size = int(len(pd_dataset) * 0.9)
valid_set_size = len(pd_dataset) - train_set_size

In [ ]:
seed = torch.Generator().manual_seed(42)
train_set, valid_set = data.random_split(
    pd_dataset, [train_set_size, valid_set_size], generator=seed
)

In [ ]:
train_loader = data.DataLoader(
    dataset=train_set,
    shuffle=True,
    batch_size=20,
    collate_fn=collate_padded_batch_fn,
    num_workers=8,
)
valid_loader = data.DataLoader(
    dataset=valid_set,
    shuffle=True,
    batch_size=20,
    collate_fn=collate_padded_batch_fn,
    num_workers=0,
)

In [ ]:
logger = TensorBoardLogger(save_dir=data_path / "tb_logs", name="besline_test")
trainer = L.Trainer(max_epochs=500, accelerator="auto", logger=logger)
trainer.fit(model=b_lstm, train_dataloaders=train_loader, val_dataloaders=valid_loader)

In [ ]:
! dirname $data_path

In [ ]:
! ls $data_path

In [ ]:
# tensorboard --host 0.0.0.0 --port 6006 --logdir=/home/jovyan/data/tbts_data/tb_logs